In [ ]:
"""
RQ4 — Governance-Filtered SHAP Analysis
==========================================
Dissertation: Predicting Revenue Growth and Cost Reduction from Business AI Adoption

PURPOSE
-------
Directly answers RQ4: "Do organisational AI governance practices influence
revenue growth and cost reduction?"

Reuses the SHAP values ALREADY computed and saved by shared_pipeline.py for
BOTH targets (no new model training or SHAP computation required — this is
the "minimal additional computational cost" step described in Section 3.7).

Filters those existing SHAP values down to just the four governance
variables:
    - ai_risk_management_score      (numeric)
    - regulatory_compliance_score   (numeric)
    - ai_ethics_committee           (categorical -> one-hot encoded)
    - data_privacy_level            (categorical -> one-hot encoded)

Produces:
  1. A governance-only comparison table (importance + direction, both targets)
  2. A governance-only tornado plot (same style as the RQ2 comparative figure)
  3. A "governance share of total explanatory power" statistic — what
     proportion of each model's total |SHAP| comes from governance features
     specifically, giving a sense of relative (not just absolute) importance
  4. Governance SHAP direction-of-effect summary for each target

Run: python governance_shap_analysis.py
Requires: shared_pipeline.py must have been run (FULL run, not quick test)
          for BOTH targets first.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════

SHAP_DIR = "pipeline_outputs/shap"
OUT_DIR = "pipeline_outputs/governance_shap"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_1 = "revenue_growth_percent"
TARGET_2 = "cost_reduction_percent"
TARGET_1_LABEL = "Revenue Growth"
TARGET_2_LABEL = "Cost Reduction"

NAVY = "#1F4E79"
AMBER = "#E8A33D"

# The four governance variables as they appear BEFORE one-hot encoding.
# ai_ethics_committee and data_privacy_level are categorical, so after
# preprocessing they will appear in feature_names as e.g.
# "ai_ethics_committee_Yes", "data_privacy_level_Medium", etc. — the matching
# below uses a "starts with" check to catch all encoded variants automatically.
GOVERNANCE_BASE_VARS = [
    "ai_risk_management_score",
    "regulatory_compliance_score",
    "ai_ethics_committee",
    "data_privacy_level",
]

# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD BOTH SAVED SHAP FILES (same files used for RQ2 — no recomputation)
# ═══════════════════════════════════════════════════════════════════════════

df = pd.read_csv("ai_company_adoption.csv")

def load_shap_data(target_column, shap_dir=SHAP_DIR):
    path = f"{shap_dir}/{target_column}_shap_values.pkl"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nCould not find '{path}'.\n"
            f"Make sure shared_pipeline.py has been run with sample_frac=None "
            f"(a FULL run, not a quick test) for target_column='{target_column}' first."
        )
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

print("=" * 80)
print("LOADING SAVED SHAP VALUES (reusing RQ2's computation — no new SHAP run)")
print("=" * 80)

data_1 = load_shap_data(TARGET_1)
data_2 = load_shap_data(TARGET_2)

print(f"✅ Loaded {TARGET_1}: {data_1['shap_values'].shape[0]:,} observations, "
      f"{len(data_1['feature_names'])} total features")
print(f"✅ Loaded {TARGET_2}: {data_2['shap_values'].shape[0]:,} observations, "
      f"{len(data_2['feature_names'])} total features")

# ═══════════════════════════════════════════════════════════════════════════
# 2. IDENTIFY WHICH ENCODED FEATURE NAMES ARE GOVERNANCE-RELATED
# ═══════════════════════════════════════════════════════════════════════════

def find_governance_features(feature_names, base_vars=GOVERNANCE_BASE_VARS):
    """Match encoded feature names (e.g. 'ai_ethics_committee_Yes') back to
    their governance base variable via prefix matching."""
    matched = []
    for fname in feature_names:
        for base in base_vars:
            if fname == base or fname.startswith(base + "_"):
                matched.append(fname)
                break
    return matched

gov_features_1 = find_governance_features(data_1["feature_names"])
gov_features_2 = find_governance_features(data_2["feature_names"])

print(f"\nGovernance-related features found for {TARGET_1_LABEL}: {gov_features_1}")
print(f"Governance-related features found for {TARGET_2_LABEL}: {gov_features_2}")

if not gov_features_1 or not gov_features_2:
    raise ValueError(
        "\n⚠ No governance features matched! Check that GOVERNANCE_BASE_VARS "
        "matches the actual column names used in shared_pipeline.py, and that "
        "the categorical governance variables were one-hot encoded as expected."
    )

# ═══════════════════════════════════════════════════════════════════════════
# 3. COMPUTE FULL-MODEL AND GOVERNANCE-ONLY SHAP SUMMARIES
# ═══════════════════════════════════════════════════════════════════════════

def summarise_shap(shap_values, feature_names, subset=None):
    """Mean |SHAP| and mean signed SHAP per feature. If `subset` is given,
    restrict to only those feature names."""
    shap_values = np.asarray(shap_values)
    df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        "mean_signed_shap": shap_values.mean(axis=0),
    })
    if subset is not None:
        df = df[df["feature"].isin(subset)]
    return df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

# Full-model totals (for computing governance's SHARE of total importance)
full_summary_1 = summarise_shap(data_1["shap_values"], data_1["feature_names"])
full_summary_2 = summarise_shap(data_2["shap_values"], data_2["feature_names"])
total_importance_1 = full_summary_1["mean_abs_shap"].sum()
total_importance_2 = full_summary_2["mean_abs_shap"].sum()

# Governance-only subsets
gov_summary_1 = summarise_shap(data_1["shap_values"], data_1["feature_names"], subset=gov_features_1)
gov_summary_2 = summarise_shap(data_2["shap_values"], data_2["feature_names"], subset=gov_features_2)

print("\n" + "=" * 80)
print(f"GOVERNANCE FEATURE IMPORTANCE — {TARGET_1_LABEL}")
print("=" * 80)
print(gov_summary_1.to_string(index=False))

print("\n" + "=" * 80)
print(f"GOVERNANCE FEATURE IMPORTANCE — {TARGET_2_LABEL}")
print("=" * 80)
print(gov_summary_2.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# 4. GOVERNANCE'S SHARE OF TOTAL MODEL EXPLANATORY POWER
# ═══════════════════════════════════════════════════════════════════════════

gov_share_1 = gov_summary_1["mean_abs_shap"].sum() / total_importance_1 * 100
gov_share_2 = gov_summary_2["mean_abs_shap"].sum() / total_importance_2 * 100

print("\n" + "=" * 80)
print("GOVERNANCE SHARE OF TOTAL MODEL EXPLANATORY POWER")
print("=" * 80)
print(f"{TARGET_1_LABEL}: governance features account for {gov_share_1:.2f}% "
      f"of total |SHAP| across all {len(data_1['feature_names'])} features")
print(f"{TARGET_2_LABEL}: governance features account for {gov_share_2:.2f}% "
      f"of total |SHAP| across all {len(data_2['feature_names'])} features")

print("\n--- Suggested dissertation text (Findings, RQ4) ---")
print(
    f"'Governance-related variables collectively account for {gov_share_1:.2f}% of total explanatory "
    f"power in the {TARGET_1_LABEL.lower()} model, compared to {gov_share_2:.2f}% in the "
    f"{TARGET_2_LABEL.lower()} model, indicating that governance practices play a "
    f"{'more' if gov_share_1 > gov_share_2 else 'less'} substantial role in predicting "
    f"{TARGET_1_LABEL.lower()} than {TARGET_2_LABEL.lower()}.'"
)

# ═══════════════════════════════════════════════════════════════════════════
# 5. MERGED GOVERNANCE COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════

gov_comparison = pd.merge(
    gov_summary_1, gov_summary_2, on="feature", how="outer",
    suffixes=(f"_{TARGET_1}", f"_{TARGET_2}")
).fillna(0)
gov_comparison = gov_comparison.sort_values(
    by=[f"mean_abs_shap_{TARGET_1}", f"mean_abs_shap_{TARGET_2}"], ascending=False
).reset_index(drop=True)

gov_comparison.to_csv(f"{OUT_DIR}/rq4_governance_shap_table.csv", index=False)
print(f"\n✅ Saved: {OUT_DIR}/rq4_governance_shap_table.csv")

# Direction interpretation (does the feature push the outcome up or down, on average?)
print("\n" + "=" * 80)
print("DIRECTION OF EFFECT — governance features")
print("=" * 80)
for _, row in gov_comparison.iterrows():
    dir_1 = "increases" if row[f"mean_signed_shap_{TARGET_1}"] > 0 else "decreases"
    dir_2 = "increases" if row[f"mean_signed_shap_{TARGET_2}"] > 0 else "decreases"
    print(f"{row['feature']:35s}  {dir_1:>9s} {TARGET_1_LABEL.lower():16s}"
          f"  |  {dir_2:>9s} {TARGET_2_LABEL.lower()}")

# ═══════════════════════════════════════════════════════════════════════════
# 6. GOVERNANCE TORNADO PLOT
# ═══════════════════════════════════════════════════════════════════════════

plot_df = gov_comparison.copy()
plot_df["combined"] = plot_df[f"mean_abs_shap_{TARGET_1}"] + plot_df[f"mean_abs_shap_{TARGET_2}"]
plot_df = plot_df.sort_values("combined", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.5)))
y_pos = np.arange(len(plot_df))
ax.barh(y_pos, -plot_df[f"mean_abs_shap_{TARGET_1}"], color=NAVY, label=TARGET_1_LABEL, height=0.6)
ax.barh(y_pos, plot_df[f"mean_abs_shap_{TARGET_2}"], color=AMBER, label=TARGET_2_LABEL, height=0.6)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["feature"], fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)

max_val = max(plot_df[f"mean_abs_shap_{TARGET_1}"].max(), plot_df[f"mean_abs_shap_{TARGET_2}"].max())
max_val = max(max_val, 0.001)  # avoid zero-width axis if all governance values are tiny
ax.set_xlim(-max_val * 1.3, max_val * 1.3)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.3f}" for x in xticks])

ax.set_xlabel("Mean |SHAP value|  (feature importance)", fontsize=10)
ax.set_title(
    f"RQ4 — Governance-Filtered SHAP Comparison\n{TARGET_1_LABEL} (left) vs {TARGET_2_LABEL} (right)",
    fontsize=13, fontweight="bold"
)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/rq4_governance_shap_tornado.png", dpi=150, bbox_inches="tight")
print(f"\n✅ Saved main RQ4 figure: {OUT_DIR}/rq4_governance_shap_tornado.png")
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# 7. GOVERNANCE SHARE — SIMPLE BAR CHART (governance vs everything else)
# ═══════════════════════════════════════════════════════════════════════════

fig2, ax2 = plt.subplots(figsize=(7, 5))
categories = [TARGET_1_LABEL, TARGET_2_LABEL]
gov_shares = [gov_share_1, gov_share_2]
other_shares = [100 - gov_share_1, 100 - gov_share_2]

x = np.arange(len(categories))
ax2.bar(x, gov_shares, color=NAVY, label="Governance features", width=0.5)
ax2.bar(x, other_shares, bottom=gov_shares, color="#D6E4F0", label="All other features", width=0.5)

for i, share in enumerate(gov_shares):
    ax2.text(i, share / 2, f"{share:.1f}%", ha="center", va="center",
              fontsize=10, fontweight="bold", color="white")

ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=10)
ax2.set_ylabel("Share of total |SHAP| (%)", fontsize=10)
ax2.set_title("RQ4 — Governance Features' Share of\nTotal Model Explanatory Power", fontsize=12, fontweight="bold")
ax2.legend(loc="upper right", fontsize=9)
ax2.set_ylim(0, 100)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/rq4_governance_share.png", dpi=150, bbox_inches="tight")
print(f"✅ Saved governance share figure: {OUT_DIR}/rq4_governance_share.png")
plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# DONE
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("RQ4 GOVERNANCE-FILTERED SHAP ANALYSIS COMPLETE")
print("=" * 80)
print(f"""
Files generated in {OUT_DIR}/:
  rq4_governance_shap_table.csv    — governance-only comparison table
  rq4_governance_shap_tornado.png  — MAIN FIGURE: governance importance, both targets
  rq4_governance_share.png         — governance's % share of total explanatory power

Governance share of total |SHAP|:
  {TARGET_1_LABEL}: {gov_share_1:.2f}%
  {TARGET_2_LABEL}: {gov_share_2:.2f}%
""")

In [ ]:
print(df['ai_ethics_committee'].value_counts(normalize=True))

In [ ]:
import pandas as pd
imp_revenue = pd.read_csv("pipeline_outputs/metrics/revenue_growth_percent_native_feature_importance.csv")
imp_cost = pd.read_csv("pipeline_outputs/metrics/cost_reduction_percent_native_feature_importance.csv")

print(imp_revenue[imp_revenue['feature'].str.contains('ethics_committee')])
print(imp_cost[imp_cost['feature'].str.contains('ethics_committee')])